In [2]:
from typing import List, Set, Tuple
import numpy as np

class PackingSolver:
    def __init__(self, cube_size: int = 5):
        self.size = cube_size
        self.cube = np.zeros((cube_size, cube_size, cube_size), dtype=int)
        # T piece with extended side defined in 3D
        self.piece = np.array([[[1,0,0,0],
                              [1,1,0,0],
                              [1,0,0,0],
                              [1,0,0,0]]])
        self.solutions = []

    def validate_solution(self) -> bool:
        """Validate that each number appears exactly 5 times and forms a valid T-shape."""
        for piece_num in range(1, 26):
            positions = np.where(self.cube == piece_num)
            if len(positions[0]) != 5:  # Check piece has exactly 5 cells
                return False
            
            # Check connectivity and T-shape
            cells = set(zip(positions[0], positions[1], positions[2]))
            if not self.is_valid_t_shape(cells):
                return False
        return True

    def is_valid_t_shape(self, cells: Set[Tuple[int, int, int]]) -> bool:
        """Check if cells form a valid T shape in any orientation."""
        if len(cells) != 5:
            return False
            
        # Find connected components
        visited = set()
        def dfs(cell):
            visited.add(cell)
            x, y, z = cell
            for dx, dy, dz in [(0,0,1), (0,0,-1), (0,1,0), (0,-1,0), (1,0,0), (-1,0,0)]:
                next_cell = (x+dx, y+dy, z+dz)
                if next_cell in cells and next_cell not in visited:
                    dfs(next_cell)
                    
        start = next(iter(cells))
        dfs(start)
        return len(visited) == 5  # All cells must be connected

    def get_rotations(self) -> List[np.ndarray]:
        rotations = []
        piece = self.piece
        # Rotate around each axis
        for axis in [(1,0), (0,2), (1,2)]:
            for k in range(4):
                piece = np.rot90(piece, k=1, axes=axis)
                rotations.append(piece)
        return list(set(map(tuple, rotations)))

    def can_place(self, pos: Tuple[int, int, int], piece: np.ndarray) -> bool:
        x, y, z = pos
        if x + piece.shape[0] > self.size or y + piece.shape[1] > self.size or z + piece.shape[2] > self.size:
            return False
        
        test_cube = self.cube[x:x+piece.shape[0], y:y+piece.shape[1], z:z+piece.shape[2]]
        return not np.any(np.logical_and(test_cube, piece))

    def place(self, pos: Tuple[int, int, int], piece: np.ndarray, value: int):
        x, y, z = pos
        mask = piece == 1
        self.cube[x:x+piece.shape[0], y:y+piece.shape[1], z:z+piece.shape[2]][mask] = value

    def solve(self, pieces_left: int = 25, piece_num: int = 1) -> bool:
        if pieces_left == 0:
            if self.validate_solution():
                self.solutions.append(np.copy(self.cube))
                return True
            return False

        for x in range(self.size):
            for y in range(self.size):
                for z in range(self.size):
                    for rot in self.get_rotations():
                        if self.can_place((x,y,z), rot):
                            self.place((x,y,z), rot, piece_num)
                            if self.solve(pieces_left - 1, piece_num + 1):
                                return True
                            self.place((x,y,z), rot, 0)
        return False

    def print_solution(self):
        if not self.solutions:
            print("No solution found")
            return
        
        solution = self.solutions[0]
        for z in range(self.size):
            print(f"\nLayer {z+1}:")
            print(solution[:,:,z])

solver = PackingSolver()
if solver.solve():
    solver.print_solution()
else:
    print("No solution exists")

TypeError: unhashable type: 'numpy.ndarray'